[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch12-benchmarking/02-do-do-tre-cho-dung.ipynb)

# Đo độ trễ cho đúng: làm nóng, phân vị, kịch bản và định luật Amdahl

Một con số độ trễ (latency) chỉ có nghĩa khi biết nó được đo thế nào. Chương 12 nêu bốn quy tắc của
thám tử hệ thống cho benchmark vi mô, ba thực hành của checkpoint 1.3, và một loạt cảnh báo: trung
bình che giấu phần đuôi, cùng một phần cứng báo những con số rất khác nhau tuỳ cách yêu cầu đến, và
định luật Amdahl chặn trần mọi tuyên bố tăng tốc cục bộ.

Sổ tay này đo thật trên CPU của bạn, với MobileNetV2, mô hình xuyên suốt của chương. Mọi thời gian ở
đây là số đo của máy đang chạy: chúng khác nhau giữa các máy, và không con số nào được đặt cạnh số của
sách. Thứ được kiểm là **hình dạng** của các kết quả.

**Bạn sẽ làm:**
1. dựng một khung kiểm thử nhỏ, ghi lại máy, rồi đo những lần chạy đầu tiên;
2. đo phân phối độ trễ: trung bình, các phân vị, và hệ số biến thiên;
3. xem trung bình che giấu phần đuôi, với một đợt thu gom rác giả lập theo kiểu Discord;
4. đo độ trễ và thông lượng (throughput) theo kích thước batch, rồi đọc ba kịch bản MLPerf từ cùng các
   số đo ấy;
5. tính lại phép tính nhẩm 1.4 về định luật Amdahl, rồi đo nó trên một đường đi có giải mã JPEG.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Khung kiểm thử: ghi lại máy, rồi làm nóng

Checkpoint 1.3 của sách nêu ba thực hành: dữ liệu đại diện, nói rõ bao nhiêu lần chạy đầu đã bị bỏ, và
cô lập, tức ghi lại máy, các tải chạy cạnh tranh và cấu hình môi trường chạy lúc đo. Ô dưới ghi lại
phần mà một sổ tay tự biết được về máy của nó.

Mô hình là MobileNetV2 của torchvision với trọng số ngẫu nhiên, 10 lớp, ảnh đầu vào 64×64 để chạy
nhanh trên CPU. Thời gian một lượt truyền xuôi không phụ thuộc giá trị của trọng số, nên không cần
huấn luyện.

In [ ]:
import gc
import io
import platform

import torchvision
from PIL import Image

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
R = 64  # cạnh ảnh đầu vào của mô hình

print(f"máy: {platform.machine()} · {platform.system()} {platform.release()} · {os.cpu_count()} lõi logic")
print(f"PyTorch {torch.__version__}, {torch.get_num_threads()} luồng · Python {platform.python_version()}")

dat_hat_giong()
mo_hinh = torchvision.models.mobilenet_v2(weights=None, num_classes=10).eval()
x1 = torch.randn(1, 3, R, R)


def bam_gio(f, lap):
    """Thời gian từng lần gọi f(), tính bằng mi-li-giây. Không làm nóng: người gọi tự quyết."""
    out = np.empty(lap)
    for i in range(lap):
        t0 = time.perf_counter()
        f()
        out[i] = 1000 * (time.perf_counter() - t0)
    return out

### Làm nóng

Quy tắc làm nóng nói không được đo những vòng khởi động nguội (cold start) như thể đó là hiệu năng ổn
định. Lần gọi đầu tiên của một mô hình PyTorch còn phải cấp phát bộ nhớ và chuẩn bị kernel cho từng
tầng.

**Dự đoán:** lần gọi đầu tiên chậm hơn lần thứ hai bao nhiêu lần?

In [ ]:
with torch.no_grad():
    dau_tien = bam_gio(lambda: mo_hinh(x1), 20)
BO_DI = 10  # số lần chạy đầu bị bỏ trước khi ghi số đo: báo con số này cùng kết quả
do_tren_may("lần gọi thứ nhất", dau_tien[0], "ms")
do_tren_may("trung vị của lần thứ 11 tới 20", np.median(dau_tien[BO_DI:]), "ms")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, 21), dau_tien, "o-", color=MAU[0], lw=1.5)
truc_log = dau_tien.max() / dau_tien.min() > 10  # lần đầu chậm hơn hẳn thì cần thang log để thấy phần còn lại
if truc_log:
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
ax.axvspan(0.5, BO_DI + 0.5, color="0.85", zorder=0)
ax.text((BO_DI + 1) / 2, 0.55, f"{BO_DI} lần đầu: bỏ đi", fontsize=8, ha="center",
        transform=ax.get_xaxis_transform())
ax.set_xticks(range(1, 21, 2))
ax.set_xlabel("lần gọi")
ax.set_ylabel("ms (thang log, đo trên máy này)" if truc_log else "ms (đo trên máy này)")
ax.set_title("Những lần gọi đầu tiên của MobileNetV2, batch 1")
plt.tight_layout()
plt.show()

## 2. Một số đo là một phân phối

Đo lại cùng một lượt truyền xuôi 600 lần thì được 600 con số khác nhau, vì nhiễu của hệ điều hành,
bộ đệm, xung nhịp và các tiến trình khác. Quy tắc phương sai của sách yêu cầu báo hệ số biến thiên,
tức độ lệch chuẩn chia cho giá trị trung bình qua các lần chạy lặp, và điều tra những giá trị vượt
dung sai riêng của giao thức. Sách nói rõ 5 phần trăm không phải ngưỡng dùng chung cho mọi tải.

Hai cách nhìn hệ số biến thiên cho hai câu trả lời khác nhau:

* **trong một lần chạy:** 600 lần gọi nối tiếp nhau, mỗi lần gọi một số đo;
* **giữa các lần chạy:** 5 lần chạy, mỗi lần 100 lần gọi, mỗi lần chạy tóm thành một trung vị.

**Dự đoán:** trung bình sẽ lớn hơn, bằng, hay nhỏ hơn trung vị?

In [ ]:
with torch.no_grad():
    mau_do = bam_gio(lambda: mo_hinh(x1), 600)
p50, p90, p95, p99 = np.percentile(mau_do, [50, 90, 95, 99])
do_tren_may("trung bình", mau_do.mean(), "ms")
for ten, gt in (("phân vị 50 (trung vị)", p50), ("phân vị 90", p90), ("phân vị 95", p95), ("phân vị 99", p99)):
    do_tren_may(ten, gt, "ms")
do_tren_may("lớn nhất", mau_do.max(), "ms")
cv_trong = mau_do.std() / mau_do.mean()
do_tren_may("hệ số biến thiên trong một lần chạy", 100 * cv_trong, "%")

with torch.no_grad():
    trung_vi_lan = np.array([np.median(bam_gio(lambda: mo_hinh(x1), 100)) for _ in range(5)])
cv_giua = trung_vi_lan.std() / trung_vi_lan.mean()
do_tren_may("hệ số biến thiên giữa trung vị của 5 lần chạy", 100 * cv_giua, "%")

import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(7.5, 3.6))
thang_log = mau_do.max() / mau_do.min() > 3  # chỉ cần thang log khi phần đuôi kéo dài
if thang_log:
    bins = np.logspace(np.log10(mau_do.min()), np.log10(mau_do.max()), 60)
else:
    bins = np.linspace(mau_do.min(), mau_do.max(), 60)
dem, _, _ = ax.hist(mau_do, bins=bins, color=MAU[0], alpha=0.75)
ax.set_ylim(0, dem.max() * 1.15)
for gt, ten, mau in ((p50, "phân vị 50", "black"), (mau_do.mean(), "trung bình", MAU[1]), (p99, "phân vị 99", MAU[3])):
    ax.axvline(gt, color=mau, lw=1.5, ls="--", label=ten)
ax.legend(loc="upper right", fontsize=8)  # ba đường dọc có thể sát nhau, nên tên của chúng nằm trong chú giải
if thang_log:
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(mticker.LogLocator(subs=(1, 2, 5)))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())
ax.set_xlabel("ms mỗi lần gọi" + (" (thang log, đo trên máy này)" if thang_log else " (đo trên máy này)"))
ax.set_ylabel("số lần gọi")
ax.set_title("600 lần gọi cùng một lượt truyền xuôi")
plt.tight_layout()
plt.show()

Phân phối độ trễ thường lệch về bên phải: phần lớn lần gọi dồn quanh trung vị, còn một số ít chậm hơn
hẳn. Vì thế trung bình thường lớn hơn trung vị, và phân vị 99 nằm xa cả hai.

Hệ số biến thiên giữa các lần chạy thường nhỏ hơn nhiều so với trong một lần chạy, vì trung vị của 100
lần gọi đã gạt đi phần lớn nhiễu. Hai con số trả lời hai câu hỏi khác nhau: con số thứ nhất nói một
yêu cầu đơn lẻ dao động thế nào, con số thứ hai nói phép đo có tái lập được không. Một báo cáo cần nói
rõ nó đang báo con số nào.

## 3. Trung bình che giấu phần đuôi

Sách đưa một phép tính về quy mô: nếu 1 phần trăm yêu cầu chậm gấp 10 lần, như 1,000 ms thay vì 100 ms,
thì cứ một triệu yêu cầu có 10,000 yêu cầu bị ảnh hưởng.

In [ ]:
ti_le_cham = sach(1, nguon="1 phần trăm · 10× · 1,000 so với 100 ms") / 100
tre_thuong = sach(100, nguon="1,000 so với 100 ms")
tre_cham = sach(1_000, nguon="1,000 so với 100 ms")
theo_sach("hệ số chậm", tre_cham / tre_thuong, sach=10, nguon="1 phần trăm · 10×")
theo_sach("số yêu cầu bị ảnh hưởng trên một triệu", ti_le_cham * 1_000_000, sach=10_000,
          nguon="10,000 trên một triệu")
trung_binh = (1 - ti_le_cham) * tre_thuong + ti_le_cham * tre_cham
print(f"trung bình chỉ tăng từ {tre_thuong} lên {trung_binh:.0f} ms ({100 * (trung_binh / tre_thuong - 1):.0f} %),"
      f" còn trung vị không đổi: {tre_thuong} ms")

### Một đợt thu gom rác mỗi 50 yêu cầu

Câu chuyện thực tế của chương kể về dịch vụ Read States của Discord: các lượt thu gom rác (garbage
collection) bắt buộc của Go quét qua một bộ nhớ đệm LRU giữ hàng chục triệu mục, cứ hai phút một
lần, và độ trễ đuôi (tail latency) ở phân vị 99 vọt lên theo đúng nhịp ấy.

Python cũng có một bộ thu gom rác cho các vòng tham chiếu, và một lượt thu gom đầy đủ phải duyệt qua mọi
đối tượng còn sống. Ô dưới dựng một bộ nhớ đệm giữ nhiều mục, đo thời gian một lượt `gc.collect()` theo
kích thước của nó, rồi phục vụ 400 yêu cầu, mỗi yêu cầu là một lượt truyền xuôi. Lần thứ hai, cứ 50 yêu
cầu thì một yêu cầu phải chờ một lượt thu gom đầy đủ, để giả lập nhịp bắt buộc của Go.

**Dự đoán:** với 2 phần trăm yêu cầu phải chờ thu gom rác, trung vị, trung bình và phân vị 99 đổi bao
nhiêu?

In [ ]:
bo_dem = {}
dung_luong, dung_gc = [], []
for den in (100_000, 200_000, 400_000):
    for i in range(len(bo_dem), den):
        bo_dem[i] = (i, str(i), [i])  # mỗi mục giữ vài đối tượng mà bộ thu gom phải duyệt
    t0 = time.perf_counter()
    gc.collect()
    dung_luong.append(len(bo_dem))
    dung_gc.append(1000 * (time.perf_counter() - t0))
    do_tren_may(f"một lượt gc.collect() với {len(bo_dem):,} mục trong bộ nhớ đệm", dung_gc[-1], "ms")

MOI = 50  # cứ MOI yêu cầu thì một lượt thu gom


def phuc_vu(so_yeu_cau, co_gc):
    tre = np.empty(so_yeu_cau)
    with torch.no_grad():
        for i in range(so_yeu_cau):
            t0 = time.perf_counter()
            if co_gc and i % MOI == MOI - 1:
                gc.collect()  # yêu cầu này đến đúng lúc bộ thu gom dừng cả chương trình
            mo_hinh(x1)
            tre[i] = 1000 * (time.perf_counter() - t0)
    return tre


khong_gc, co_gc = phuc_vu(400, False), phuc_vu(400, True)
for ten, tre in (("không thu gom", khong_gc), (f"thu gom mỗi {MOI} yêu cầu", co_gc)):
    for nhan, gt in (("trung vị", np.median(tre)), ("trung bình", tre.mean()), ("phân vị 99", np.percentile(tre, 99))):
        do_tren_may(f"{ten}: {nhan}", gt, "ms")
del bo_dem
gc.collect()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2.2, 1]})
ax1.plot(co_gc, color=MAU[1], lw=1, label=f"thu gom mỗi {MOI} yêu cầu")
ax1.plot(khong_gc, color=MAU[0], lw=1, label="không thu gom")
ax1.set_yscale("log")
ax1.set_xlabel("yêu cầu thứ")
ax1.set_ylabel("ms (thang log, đo trên máy này)")
ax1.set_title("Độ trễ từng yêu cầu")
ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=8)
nhom = ["trung vị", "trung bình", "phân vị 99"]
gia_tri = [[np.median(t), t.mean(), np.percentile(t, 99)] for t in (khong_gc, co_gc)]
vi_tri = np.arange(3)
ax2.bar(vi_tri - 0.2, gia_tri[0], 0.4, color=MAU[0], label="không thu gom")
ax2.bar(vi_tri + 0.2, gia_tri[1], 0.4, color=MAU[1], label="có thu gom")
ax2.set_yscale("log")
ax2.set_xticks(vi_tri, nhom)
ax2.set_ylabel("ms (thang log)")
ax2.set_title("Ba cách tóm tắt")
ax2.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Trung vị gần như không đổi, trung bình tăng nhưng ít hơn nhiều, còn phân vị 99 nhảy vọt và mang gần đúng
độ dài của một lượt thu gom. Một bảng chỉ báo trung bình sẽ nói dịch vụ vẫn ổn. Thời gian dừng lớn dần theo số
đối tượng còn sống, và bộ nhớ đệm của Discord có hàng chục triệu mục.

Sách rút ra hai bài học. Độ trễ trung bình không mô tả được trải nghiệm khi mục tiêu dịch vụ đặt ở phần
đuôi. Và nguồn gốc của phần đuôi thường nằm ở kiến trúc, như một môi trường chạy có thu gom rác, nên
không tối ưu mô hình nào gỡ được: ở đây mô hình không đổi, chỉ môi trường chạy quanh nó đổi.

## 4. Batch, thông lượng và ba kịch bản MLPerf

Gom nhiều ảnh vào một batch làm mỗi ảnh rẻ đi, nhưng mỗi yêu cầu phải chờ cả batch xong. Ô dưới đo thời
gian một batch ở bảy kích thước, sau khi làm nóng từng kích thước.

**Dự đoán:** từ batch 1 lên batch 64, thông lượng tăng bao nhiêu lần?

In [ ]:
cac_batch = np.array([1, 2, 4, 8, 16, 32, 64])
thoi_gian_batch = []
with torch.no_grad():
    for b in cac_batch:
        xb = torch.randn(int(b), 3, R, R)
        mo_hinh(xb)  # làm nóng kích thước này
        thoi_gian_batch.append(np.median(bam_gio(lambda: mo_hinh(xb), max(5, 120 // int(b)))))
thoi_gian_batch = np.array(thoi_gian_batch)
thong_luong = 1000 * cac_batch / thoi_gian_batch  # ảnh mỗi giây
for b, t, q in zip(cac_batch, thoi_gian_batch, thong_luong):
    do_tren_may(f"batch {b:>2}: một batch mất", t, f"ms, tức {q:,.0f} ảnh/giây")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.loglog(cac_batch, thoi_gian_batch, "o-", color=MAU[0], lw=2)
ax1.set_ylabel("ms cho cả batch")
ax1.set_title("Độ trễ của một batch")
ax2.semilogx(cac_batch, thong_luong, "o-", color=MAU[2], lw=2)
ax2.set_ylim(0, thong_luong.max() * 1.15)
ax2.set_ylabel("ảnh mỗi giây")
ax2.set_title("Thông lượng")
for ax in (ax1, ax2):
    ax.set_xticks(cac_batch, [str(b) for b in cac_batch])
    ax.set_xlabel("kích thước batch")
fig.suptitle("MobileNetV2, ảnh 64×64 (đo trên máy này)", fontsize=10)
plt.tight_layout()
plt.show()

### Cùng phần cứng, ba con số

MLPerf Inference định nghĩa các kịch bản theo **cách yêu cầu đến**, và mỗi kịch bản báo một chỉ số riêng:

* **SingleStream:** từng yêu cầu một, yêu cầu sau chỉ đến khi kết quả trước đã được dùng. Chỉ số là
  độ trễ ở phân vị 90.
* **Offline:** mọi đầu vào có sẵn từ đầu. Chỉ số là thông lượng thuần, với batch lớn nhất.
* **Server:** yêu cầu đến ngẫu nhiên theo phân phối Poisson. Chỉ số là số truy vấn mỗi giây mà vẫn
  giữ được ràng buộc độ trễ ở phân vị 99.

Hai kịch bản đầu đọc thẳng từ các số đo ở trên. Kịch bản Server cần một hàng đợi: ô dưới mô phỏng một
máy chủ một luồng, yêu cầu đến theo Poisson, mỗi khi rảnh thì gom mọi yêu cầu đang chờ thành một batch
tối đa 64 (gom batch động, dynamic batching), với thời gian phục vụ nội suy từ bảng đo ở trên. Giới hạn
độ trễ ở phân vị 99 là giả định của sổ tay: 10 lần thời gian một ảnh đơn lẻ, để kết quả cùng hình dạng
trên mọi máy.

In [ ]:
PV_SINGLE = sach(90, nguon="phân vị 90 · phân vị 99")
PV_SERVER = sach(99, nguon="phân vị 90 · phân vị 99")
single_stream = np.percentile(mau_do, PV_SINGLE)
offline = thong_luong.max()
GIOI_HAN = 10 * thoi_gian_batch[0]  # giả định: ràng buộc ở phân vị 99, tính bằng ms
do_tren_may("SingleStream: độ trễ ở phân vị 90", single_stream, "ms")
do_tren_may("Offline: thông lượng ở batch lớn nhất", offline, "ảnh/giây")


import bisect

# Thời gian phục vụ (giây) của một batch b, nội suy tuyến tính từ bảng đo ở trên.
PHUC_VU = [0.0] + [float(np.interp(b, cac_batch, thoi_gian_batch)) / 1000 for b in range(1, 65)]


def hang_doi(den, b_max):
    """Một máy chủ một luồng, gom batch động tham lam. Trả về độ trễ (ms) của từng yêu cầu."""
    den_ds, n = list(den), len(den)
    xong = np.empty(n)
    t, i = 0.0, 0
    while i < n:
        t = max(t, den_ds[i])  # hàng trống thì chờ yêu cầu kế tiếp
        j = max(i + 1, min(i + b_max, bisect.bisect_right(den_ds, t)))
        t += PHUC_VU[j - i]
        xong[i:j] = t
        i = j
    return 1000 * (xong - np.asarray(den))


def poisson(lam, n=10_000, seed=SEED):
    return np.cumsum(np.random.default_rng(seed).exponential(1 / lam, n))


phan_tai = np.linspace(0.1, 0.95, 18)
duong = {}
for b_max in (1, 64):
    duong[b_max] = np.array([np.percentile(hang_doi(poisson(f * offline), b_max), PV_SERVER) for f in phan_tai])
    dat = phan_tai[duong[b_max] <= GIOI_HAN]
    q = dat.max() * offline if len(dat) else 0.0
    do_tren_may(f"Server, batch tối đa {b_max:>2}: số truy vấn mỗi giây còn giữ được phân vị 99", q, "truy vấn/giây")
server = phan_tai[duong[64] <= GIOI_HAN].max() * offline if np.any(duong[64] <= GIOI_HAN) else 0.0
assert server <= offline  # không cách gom batch nào vượt được thông lượng của batch lớn nhất
do_tren_may("Server chia cho Offline", server / offline)

fig, ax = plt.subplots(figsize=(7.5, 4))
for b_max, mau in ((1, MAU[0]), (64, MAU[1])):
    ax.semilogy(phan_tai * offline, duong[b_max], "o-", color=mau, lw=2, ms=4,
                label=f"Server, gom batch tối đa {b_max}")
ax.axhline(GIOI_HAN, color="0.3", lw=1, ls=":", label="giới hạn phân vị 99 (giả định)")
ax.axvline(offline, color=MAU[2], lw=1.5, ls="--")
ax.text(offline, ax.get_ylim()[0] * 1.5, "Offline ", color=MAU[2], fontsize=8, ha="right")
ax.set_xlabel("số truy vấn mỗi giây đưa vào")
ax.set_ylabel("độ trễ ở phân vị 99 (ms, thang log)")
ax.set_title("Kịch bản Server từ cùng các số đo (đo trên máy này, rồi mô phỏng)", fontsize=10)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Cùng một mô hình trên cùng một CPU cho ba con số: một độ trễ cho SingleStream, một thông lượng cho
Offline, và một số truy vấn mỗi giây thấp hơn cho Server, vì ràng buộc phân vị 99 và phụ trội xếp hàng
không cho dùng batch lớn nhất. Gom batch động nâng được số truy vấn mỗi giây của Server so với xử lý
từng yêu cầu, nhưng không bao giờ vượt Offline.

Gần trần, độ trễ đuôi tăng vọt. Đó là lý do sách đưa ví dụ một hệ thống đạt 800 truy vấn mỗi giây khi đo
cô lập chỉ duy trì được 400 tới 500 truy vấn mỗi giây dưới tải sản xuất, ở mức sử dụng (utilization)
90 phần trăm:

In [ ]:
co_lap = sach(800, nguon="800 → 400–500")
duoi_tai_cao, duoi_tai_thap = sach(500, nguon="800 → 400–500"), sach(400, nguon="800 → 400–500")
muc_su_dung = sach(90, nguon="800 → 400–500 · 90 phần trăm")
theo_sach("mức giảm khi còn 500 truy vấn/giây (%)", 100 * (1 - duoi_tai_cao / co_lap), sach=37.5,
          nguon="37.5–50 phần trăm")
theo_sach("mức giảm khi còn 400 truy vấn/giây (%)", 100 * (1 - duoi_tai_thap / co_lap), sach=50,
          nguon="37.5–50 phần trăm")
print(f"sách nêu ví dụ này ở mức sử dụng {muc_su_dung} %, do tranh chấp hàng đợi và thu gom rác")

### Cùng một tốc độ trung bình, ba kiểu đến

Sách cảnh báo một hệ thống benchmark ở 1,000 truy vấn mỗi giây đều đặn có thể sụp khi lưu lượng đột biến
lên 10,000, vì con số thông lượng đã ngầm giả định yêu cầu đến đều. Ô dưới giữ nguyên tốc độ trung bình,
ở 60 phần trăm Offline, và chỉ đổi cách yêu cầu đến: đều nhau, Poisson, hay từng chùm 20 yêu cầu cùng lúc.

In [ ]:
lam = 0.6 * offline
n_yc = 10_000
kieu_den = {
    "đều nhau": np.arange(1, n_yc + 1) / lam,
    "Poisson": poisson(lam, n_yc),
    "từng chùm 20": np.repeat(np.cumsum(np.random.default_rng(SEED).exponential(20 / lam, n_yc // 20)), 20),
}
for ten, den in kieu_den.items():
    den *= (n_yc / lam) / den[-1]  # co giãn để cả ba có đúng một tốc độ trung bình
    assert math.isclose(len(den) / den[-1], lam)
    do_tren_may(f"{ten}: độ trễ ở phân vị 99", np.percentile(hang_doi(den, 64), PV_SERVER), "ms")

## 5. Định luật Amdahl: phần không được tăng tốc đặt trần

Phép tính nhẩm 1.4 đặt một chuỗi thị giác tốn 8 ms cho tiền xử lý, gồm giải mã JPEG, đổi cỡ và chuẩn hoá,
cùng 10 ms cho suy luận (inference). Nếu chỉ suy luận được tối ưu nhanh gấp 5 lần, độ trễ đầu cuối cải
thiện bao nhiêu? Định luật Amdahl nói nếu tiền xử lý chiếm tỉ lệ $f$ của tổng độ trễ, thì dù suy luận
nhanh vô hạn, tăng tốc tối đa cũng chỉ là $1/f$:

$$S = \frac{1}{f + (1-f)/s} \le \frac{1}{f}.$$

In [ ]:
tien_xu_ly = sach(8, nguon="8 ms · 10 ms · 5×")
suy_luan = sach(10, nguon="8 ms · 10 ms · 5×")
nhanh = sach(5, nguon="8 ms · 10 ms · 5×")


def amdahl(f, s):
    """Tăng tốc đầu cuối khi phần 1 − f nhanh lên s lần, còn phần f giữ nguyên."""
    return 1 / (f + (1 - f) / s)


theo_sach("suy luận sau tối ưu (ms)", suy_luan / nhanh, sach=2, nguon="5× · 2 ms")
theo_sach("tổng trước (ms)", tien_xu_ly + suy_luan, sach=18, nguon="18 → 10 ms")
theo_sach("tổng sau (ms)", tien_xu_ly + suy_luan / nhanh, sach=10, nguon="18 → 10 ms")
theo_sach("tăng tốc đầu cuối", (tien_xu_ly + suy_luan) / (tien_xu_ly + suy_luan / nhanh), sach=1.8,
          nguon="18 → 10 ms · 1.8×")
f = tien_xu_ly / (tien_xu_ly + suy_luan)
theo_sach("phần của tiền xử lý (%)", 100 * f, sach=44.4, nguon="1.8× · 44.4 phần trăm")
theo_sach("trần 1/f", 1 / f, sach=2.25, nguon="44.4 phần trăm · 2.25×")
assert math.isclose(amdahl(f, nhanh), 1.8)

# Nguyên tắc thứ ba của chương: tăng tốc 3 lần cho chặng mô hình 10 ms trong một yêu cầu 50 ms.
nhanh_k = sach(3, nguon="3× · 10 ms · 50 ms")
chang = sach(10, nguon="3× · 10 ms · 50 ms")
yeu_cau = sach(50, nguon="3× · 10 ms · 50 ms")
sau = yeu_cau - chang + chang / nhanh_k
theo_sach("yêu cầu sau khi tăng tốc chặng mô hình (ms)", sau, sach=43, nguon="≈43 ms", tol=0.5)
theo_sach("tăng tốc đầu cuối", yeu_cau / sau, sach=1.2, nguon="≈1.2× · ≈43 ms", tol=0.05)

# Bài học thứ năm của phần tổng kết: tiền xử lý 50 % thì mô hình nhanh vô hạn cũng chỉ được 2 lần.
f_tk = sach(50, nguon="50 phần trăm · 2× · 4×") / 100
theo_sach("trần khi tiền xử lý chiếm 50 %", 1 / f_tk, sach=2, nguon="50 phần trăm · 2× · 4×")

### Bảy thành phần của một yêu cầu

Bảng 9 của sách đưa một phân rã minh hoạ, và sách nói rõ các khoảng ấy là minh hoạ chứ không phổ quát.
Cộng các đầu thấp với nhau, rồi các đầu cao, cho thấy suy luận mô hình, con số mà nhà cung cấp gọi là
benchmark, chiếm bao nhiêu phần của cả yêu cầu, và vì thế một mô hình nhanh vô hạn mua được bao nhiêu.

In [ ]:
thanh_phan = {  # (đầu thấp, đầu cao), ms; hàng đợi "0 tới hơn 1,000" lấy đúng 1,000
    "khứ hồi mạng": (sach(10, nguon="10–100 · 0.1–1"), sach(100, nguon="10–100 · 0.1–1")),
    "phân tích yêu cầu": (sach(0.1, nguon="10–100 · 0.1–1"), sach(1, nguon="10–100 · 0.1–1")),
    "tiền xử lý": (sach(1, nguon="0.1–1 · 1–50"), sach(50, nguon="0.1–1 · 1–50")),
    "chờ trong hàng đợi": (sach(0, nguon="0 tới hơn 1,000"), sach(1_000, nguon="0 tới hơn 1,000")),
    "suy luận mô hình": (sach(5, nguon="5–100 · 0.5–10"), sach(100, nguon="5–100 · 0.5–10")),
    "hậu xử lý": (sach(0.5, nguon="5–100 · 0.5–10"), sach(10, nguon="5–100 · 0.5–10")),
    "tuần tự hóa": (sach(0.1, nguon="0.5–10 · 0.1–1 ms"), sach(1, nguon="0.5–10 · 0.1–1 ms")),
}
for dau, ten_dau in ((0, "đầu thấp"), (1, "đầu cao")):
    tong = sum(v[dau] for v in thanh_phan.values())
    mo_hinh_ms = thanh_phan["suy luận mô hình"][dau]
    print(f"{ten_dau}: cả yêu cầu {tong:,.1f} ms, mô hình chiếm {100 * mo_hinh_ms / tong:.1f} %,"
          f" mô hình nhanh vô hạn cho tối đa {tong / (tong - mo_hinh_ms):.2f} lần")

Ngay ở đầu thấp, mô hình chỉ chiếm gần một phần ba yêu cầu. Ở đầu cao, hàng đợi lấn át tất cả, và đó
chính là điều sách nói: dưới tải, thành phần duy nhất mà benchmark đo bị lấn át bởi một thành phần nó
không bao giờ thấy.

Hình dưới vẽ định luật Amdahl cho ba tỉ lệ của phần không được tăng tốc, với các điểm sách tính.

In [ ]:
cac_s = np.logspace(0, 2, 200)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for f_, mau, nhan in ((1 - chang / yeu_cau, MAU[0], "f = 0.8 (chặng mô hình 10 trong 50 ms)"),
                      (f, MAU[1], "f = 0.444 (phép tính nhẩm 1.4)"), (f_tk, MAU[2], "f = 0.5 (phần tổng kết)")):
    ax.semilogx(cac_s, amdahl(f_, cac_s), color=mau, lw=2, label=nhan)
    ax.axhline(1 / f_, color=mau, lw=1, ls=":")
    ax.text(95, 1 / f_ + 0.025, f"trần {1 / f_:.3g} lần", color=mau, fontsize=8, ha="right")
for s_, f_, chu, lech, canh in ((nhanh, f, f"5 lần: {amdahl(f, nhanh):.1f} lần", (-8, 6), "right"),
                                (nhanh_k, 1 - chang / yeu_cau,
                                 f"3 lần: {amdahl(1 - chang / yeu_cau, nhanh_k):.2f} lần", (8, -14), "left")):
    ax.plot(s_, amdahl(f_, s_), "o", color="black", ms=5)
    ax.annotate(chu, (s_, amdahl(f_, s_)), xytext=lech, textcoords="offset points", fontsize=8, ha=canh)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_ylim(0.9, 2.45)
ax.set_xlabel("tăng tốc của riêng phần được tối ưu, s (thang log)")
ax.set_ylabel("tăng tốc đầu cuối, S")
ax.set_title("Định luật Amdahl: phần còn lại đặt trần")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

### Đo Amdahl trên một đường đi thật

Ô dưới dựng 30 khung hình camera 640×480 dạng JPEG, mỗi khung có một chữ số MNIST trên nền nhiễu, rồi
phục vụ từng khung qua bốn chặng: giải mã JPEG, đổi cỡ về 64×64, chuẩn hoá, và MobileNetV2. Sau đó lần lượt
thử hai cách tối ưu:

* **thay mô hình:** một CNN ba tầng rất nhỏ thay cho MobileNetV2, đứng thay cho một mô hình đã nén hay
  chưng cất. Ở đây chỉ quan tâm thời gian, không quan tâm độ chính xác;
* **tối ưu tiền xử lý:** chế độ `draft` của bộ giải mã JPEG, giải mã thẳng ở độ phân giải thấp hơn thay vì
  giải mã đủ rồi mới thu nhỏ.

**Dự đoán:** nếu mô hình nhỏ nhanh hơn MobileNetV2 hàng chục lần, cả đường đi nhanh hơn bao nhiêu lần?

In [ ]:
tap_kt = torchvision.datasets.MNIST(DATA, train=False, download=True)
rng = np.random.default_rng(SEED)


def khung_hinh(i):
    nen = (rng.random((480, 640, 3)) * 60 + np.array([90, 110, 70])).astype(np.uint8)
    chu_so = np.array(Image.fromarray(tap_kt.data[i].numpy()).resize((360, 360), Image.BILINEAR))
    nen[60:420, 140:500] = np.maximum(nen[60:420, 140:500], chu_so[..., None])
    buf = io.BytesIO()
    Image.fromarray(nen).save(buf, format="JPEG", quality=90)
    return buf.getvalue()


jpeg = [khung_hinh(i) for i in range(30)]
TB = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
DL = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def giai_ma(b, draft):
    anh = Image.open(io.BytesIO(b))
    if draft:
        anh.draft("RGB", (2 * R, 2 * R))  # bộ giải mã bỏ bớt hệ số, ra ảnh nhỏ hơn ngay từ đầu
    return anh.convert("RGB")


def chuan_hoa(anh):
    return ((torch.from_numpy(np.array(anh)).permute(2, 0, 1).float() / 255 - TB) / DL).unsqueeze(0)


nho = torch.nn.Sequential(torch.nn.Conv2d(3, 8, 3, stride=2, padding=1), torch.nn.ReLU(),
                          torch.nn.Conv2d(8, 16, 3, stride=2, padding=1), torch.nn.ReLU(),
                          torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten(), torch.nn.Linear(16, 10)).eval()


def do_duong_di(mang, draft, lap=3):
    """Thời gian trung bình (ms) của bốn chặng, trên mọi khung hình, lặp `lap` lượt sau một lượt làm nóng."""
    tong = np.zeros(4)
    with torch.no_grad():
        for luot in range(lap + 1):
            for b in jpeg:
                t0 = time.perf_counter()
                anh = giai_ma(b, draft)
                t1 = time.perf_counter()
                anh = anh.resize((R, R), Image.BILINEAR)
                t2 = time.perf_counter()
                x = chuan_hoa(anh)
                t3 = time.perf_counter()
                y = mang(x)
                t4 = time.perf_counter()
                if luot:  # lượt đầu là làm nóng
                    tong += [t1 - t0, t2 - t1, t3 - t2, t4 - t3]
    assert y.shape == (1, 10)
    return 1000 * tong / (lap * len(jpeg))


CHANG = ["giải mã JPEG", "đổi cỡ", "chuẩn hoá", "mô hình"]
cau_hinh = {"gốc": do_duong_di(mo_hinh, False), "mô hình nhỏ": do_duong_di(nho, False),
            "mô hình nhỏ + giải mã draft": do_duong_di(nho, True)}
for ten, t in cau_hinh.items():
    do_tren_may(f"{ten}: cả đường đi", t.sum(), "ms")

goc, nho_t = cau_hinh["gốc"], cau_hinh["mô hình nhỏ"]
f_do = goc[:3].sum() / goc.sum()   # phần không phải mô hình, đo được
s_do = goc[3] / nho_t[3]           # mô hình nhỏ nhanh hơn bao nhiêu lần, đo được
do_tren_may("phần tiền xử lý trong đường đi gốc", 100 * f_do, "%")
do_tren_may("mô hình nhỏ nhanh hơn MobileNetV2", s_do, "lần")
do_tren_may("Amdahl dự báo cho cả đường đi", amdahl(f_do, s_do), "lần")
do_tren_may("đo được cho cả đường đi", goc.sum() / nho_t.sum(), "lần")
do_tren_may("trần 1/f của đường đi gốc", 1 / f_do, "lần")
do_tren_may("thêm giải mã draft: cả đường đi nhanh hơn gốc", goc.sum() / cau_hinh["mô hình nhỏ + giải mã draft"].sum(), "lần")

fig, ax = plt.subplots(figsize=(8.5, 2.9))
for j, (ten, t) in enumerate(cau_hinh.items()):
    trai = 0
    for i, (chang, gt) in enumerate(zip(CHANG, t)):
        ax.barh(j, gt, left=trai, color=MAU[i], height=0.55, edgecolor="white", lw=1.5, label=chang if j == 0 else None)
        trai += gt
    ax.text(trai + 0.1, j, f"{trai:.1f} ms", va="center", fontsize=8)
ax.set_yticks(range(len(cau_hinh)), list(cau_hinh))
ax.invert_yaxis()
ax.set_xlim(0, goc.sum() * 1.15)
ax.set_xlabel("ms mỗi khung hình (đo trên máy này)")
ax.set_title("Bốn chặng của một yêu cầu, trước và sau hai cách tối ưu")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

Dự báo của Amdahl, tính từ hai số đo, khớp gần với thời gian đo được của cả đường đi, vì thời gian các chặng
cộng lại với nhau. Mô hình nhỏ nhanh hơn rất nhiều lần, còn cả đường đi chỉ tiến sát trần $1/f$. Sau đó,
mọi phần lợi còn lại nằm ở tiền xử lý: tối ưu bộ giải mã JPEG tiếp tục rút ngắn đường đi, trong khi làm mô
hình nhanh thêm nữa gần như không đổi được gì.

Đây là kết luận của sách: benchmark toàn diện hoặc phải đo cả tiền xử lý, hoặc phải nói rõ rằng mức tăng
tốc báo cáo chỉ áp dụng cho phần suy luận.

## Thử thêm

1. Ở phần 3, đổi `MOI` thành 200 (0.5 phần trăm yêu cầu gặp thu gom rác). Phân vị 99 còn bắt được đợt
   thu gom không? Phân vị 99.9 thì sao?
2. Ở phần 4, đổi `GIOI_HAN` thành `3 * thoi_gian_batch[0]` hoặc `30 * thoi_gian_batch[0]`. Kịch bản Server
   giữ được bao nhiêu phần của Offline?
3. Ở phần 5, đổi `R` ở ô đầu thành 128 rồi chạy lại từ đầu. Phần của mô hình trong đường đi gốc tăng hay
   giảm, và trần $1/f$ đổi thế nào?

## Tóm lại

* Một số đo độ trễ là một phân phối: bỏ các lần chạy đầu, báo số lần bỏ, rồi báo trung vị, phân vị 99 và
  hệ số biến thiên, kèm cấu hình máy.
* Trung bình che giấu phần đuôi: 2 phần trăm yêu cầu gặp thu gom rác gần như không đổi trung vị, nhưng kéo
  phân vị 99 lên bằng cả một lượt thu gom. Nguồn của phần đuôi thường là kiến trúc, không phải mô hình.
* Cùng phần cứng báo ba con số theo cách yêu cầu đến: SingleStream, Offline, và Server thấp hơn Offline
  vì ràng buộc phân vị 99 và phụ trội xếp hàng.
* Định luật Amdahl: 8 ms tiền xử lý cộng 10 ms suy luận, suy luận nhanh 5 lần thì cả đường đi chỉ nhanh 1.8
  lần, trần 2.25 lần. Đo cả đường đi, hoặc nói rõ con số chỉ cho phần suy luận.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Đo chuẩn hiệu năng — khi lời hứa tối ưu gặp số đo](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch12-benchmarking/), dựng từ chương
[*Benchmarking*](https://mlsysbook.ai/vol1/benchmarking/benchmarking.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_12_perf_bench.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.